In [ ]:
# ============================================
# TILED BRAILLE DETECTION + VISUALIZATION
# ============================================
#
# Pipeline:
# 1. Load large image
# 2. Slice into overlapping tiles
# 3. Run YOLO on each tile
# 4. Convert detections back to original coords
# 5. Draw RED dots on original image
# 6. Save final full-size visualization
#
# ============================================

from ultralytics import YOLO
from PIL import Image
import cv2
import numpy as np
import os

# ============================================
# CONFIG
# ============================================

MODEL_PATH = "app/models/best_01.pt"

SOURCE_DIR = r"C:/Users/rohan/Downloads/Braille Dots/cropped"

DEST_DIR = r"C:/Users/rohan/Downloads/Braille Dots/cropped/results"

# Tile settings
TILE_SIZE = 640
OVERLAP_RATIO = 0.2

# YOLO confidence
CONF = 0.25

# Dot visualization
DOT_RADIUS = 3
DOT_COLOR = (0, 0, 255)  # Red in BGR

# ============================================
# CREATE OUTPUT DIRECTORY
# ============================================

os.makedirs(DEST_DIR, exist_ok=True)

# ============================================
# LOAD MODEL
# ============================================

print("Loading YOLO model...")
model = YOLO(MODEL_PATH)

# ============================================
# VALID IMAGE EXTENSIONS
# ============================================

VALID_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
)

# ============================================
# GET IMAGE FILES
# ============================================

image_files = []

for file_name in os.listdir(SOURCE_DIR):

    full_path = os.path.join(SOURCE_DIR, file_name)

    if not os.path.isfile(full_path):
        continue

    if not file_name.lower().endswith(VALID_EXTENSIONS):
        continue

    image_files.append(full_path)

print(f"Found {len(image_files)} images")

# ============================================
# PROCESS EACH IMAGE
# ============================================

for img_idx, img_path in enumerate(image_files, start=1):

    print("\n====================================")
    print(f"[{img_idx}/{len(image_files)}]")
    print("Processing:", img_path)

    # ========================================
    # LOAD ORIGINAL IMAGE
    # ========================================

    original_pil = Image.open(img_path).convert("RGB")

    original_np = np.array(original_pil)

    # Convert RGB -> BGR for OpenCV
    original_cv = cv2.cvtColor(original_np, cv2.COLOR_RGB2BGR)

    width, height = original_pil.size

    # ========================================
    # TILE SETTINGS
    # ========================================

    stride = int(TILE_SIZE * (1 - OVERLAP_RATIO))

    total_detections = 0

    # ========================================
    # SLIDE OVER IMAGE
    # ========================================

    for top in range(0, height, stride):

        for left in range(0, width, stride):

            right = min(left + TILE_SIZE, width)
            bottom = min(top + TILE_SIZE, height)

            # Crop tile
            tile = original_pil.crop((left, top, right, bottom))

            tile_np = np.array(tile)

            # ====================================
            # YOLO PREDICTION
            # ====================================

            results = model.predict(
                source=tile_np,
                conf=CONF,
                verbose=False
            )

            # ====================================
            # PROCESS DETECTIONS
            # ====================================

            for result in results:

                boxes = result.boxes.xyxy.cpu().numpy()

                total_detections += len(boxes)

                for box in boxes:

                    x1, y1, x2, y2 = box[:4]

                    # Detection center inside tile
                    cx_tile = int((x1 + x2) / 2)
                    cy_tile = int((y1 + y2) / 2)

                    # Convert to ORIGINAL IMAGE coords
                    cx_global = left + cx_tile
                    cy_global = top + cy_tile

                    # =================================
                    # DRAW RED DOT
                    # =================================

                    cv2.circle(
                        original_cv,
                        (cx_global, cy_global),
                        radius=DOT_RADIUS,
                        color=DOT_COLOR,
                        thickness=-1,
                        lineType=cv2.LINE_AA
                    )

    # ========================================
    # SAVE FINAL IMAGE
    # ========================================

    filename = os.path.basename(img_path)

    save_path = os.path.join(DEST_DIR, filename)

    cv2.imwrite(save_path, original_cv)

    print(f"✅ Saved: {save_path}")
    print(f"🔴 Total detections: {total_detections}")

print("\n====================================")
print("✅ ALL IMAGES COMPLETED")
print("====================================")

Loading YOLO model...
Found 114 images

[1/114]
Processing: C:/Users/rohan/Downloads/Braille Dots/cropped\001.jpg
✅ Saved: C:/Users/rohan/Downloads/Braille Dots/cropped/results\001.jpg
🔴 Total detections: 2562

[2/114]
Processing: C:/Users/rohan/Downloads/Braille Dots/cropped\002.jpg
✅ Saved: C:/Users/rohan/Downloads/Braille Dots/cropped/results\002.jpg
🔴 Total detections: 2675

[3/114]
Processing: C:/Users/rohan/Downloads/Braille Dots/cropped\003.jpg
✅ Saved: C:/Users/rohan/Downloads/Braille Dots/cropped/results\003.jpg
🔴 Total detections: 2588

[4/114]
Processing: C:/Users/rohan/Downloads/Braille Dots/cropped\004.jpg
✅ Saved: C:/Users/rohan/Downloads/Braille Dots/cropped/results\004.jpg
🔴 Total detections: 2529

[5/114]
Processing: C:/Users/rohan/Downloads/Braille Dots/cropped\005.jpg
✅ Saved: C:/Users/rohan/Downloads/Braille Dots/cropped/results\005.jpg
🔴 Total detections: 2596

[6/114]
Processing: C:/Users/rohan/Downloads/Braille Dots/cropped\006.jpg
✅ Saved: C:/Users/rohan/Downloa